In [3]:
# last
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
    get_response_synthesizer,
    StorageContext,
    load_index_from_storage,
)
from llama_parse import LlamaParse
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.evaluation import (
    RelevancyEvaluator,
    CorrectnessEvaluator,
)
import pandas as pd
import asyncio
import json
import time
import nest_asyncio
import os
from types import SimpleNamespace
from llama_index.core.llama_dataset import (
    LabelledRagDataExample,
)
import itertools

# Apply nested asyncio for notebook execution
nest_asyncio.apply()


class RAGEvaluator:
    def __init__(
        self,
        data_directory="data",
        dataset_file="./rag_dataset.json",
        index_persist_dir="./indexes",
    ):
        # Configure settings
        self._configure_settings()

        # Initialize parser and file extractor
        self.parser = LlamaParse(result_type="markdown")
        self.file_extractor = {".pdf": self.parser}

        # Set data directory and persist directory
        self.data_directory = data_directory
        self.index_persist_dir = index_persist_dir

        # Ensure index directory exists
        os.makedirs(self.index_persist_dir, exist_ok=True)

        # Load dataset
        self.rag_dataset = self._load_dataset(dataset_file)

        # Initialize evaluators
        self.relevancy_evaluator = RelevancyEvaluator()
        self.correctness_evaluator = CorrectnessEvaluator()

        # Initialize results DataFrame
        self.eval_results_df = pd.DataFrame()

        # Initialize summary results DataFrame for different parameter combinations
        self.summary_results_df = pd.DataFrame()

    def _configure_settings(self):
        """Configure global settings for embedding model and LLM"""
        Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-zh-v1.5")
        Settings.llm = Ollama(
            model="llama3.1:latest", request_timeout=60.0, temperature=0.3
        )

    def _load_documents(self, data_directory):
        """Load documents from the specified directory"""
        return SimpleDirectoryReader(
            data_directory, file_extractor=self.file_extractor
        ).load_data()

    def _load_dataset(self, dataset_file):
        """Load the RAG dataset from a JSON file"""
        with open(dataset_file, "r") as f:
            # Convert dictionary to object with attribute access
            return json.load(f, object_hook=lambda d: SimpleNamespace(**d))

    def _get_index_persist_path(self, chunk_size, chunk_overlap):
        """Get the directory path for persisting the index"""
        # 使用人類可讀的格式
        index_dir = f"chunk_size{chunk_size}_overlap{chunk_overlap}"
        return os.path.join(self.index_persist_dir, index_dir)

    def _create_or_load_index(self, chunk_size, chunk_overlap):
        """Create a new index or load an existing one based on chunk parameters"""
        index_persist_path = self._get_index_persist_path(chunk_size, chunk_overlap)

        if os.path.exists(index_persist_path):
            print(
                f"Loading existing index for chunk_size={chunk_size}, chunk_overlap={chunk_overlap}"
            )
            # Load existing index
            storage_context = StorageContext.from_defaults(
                persist_dir=index_persist_path
            )
            index = load_index_from_storage(storage_context)
        else:
            print(
                f"Creating new index for chunk_size={chunk_size}, chunk_overlap={chunk_overlap}"
            )
            documents = self._load_documents(self.data_directory)
            # Create text splitter with the specified chunk size and overlap
            text_splitter = SentenceSplitter(
                chunk_size=chunk_size, chunk_overlap=chunk_overlap
            )

            # Create vector index
            index = VectorStoreIndex.from_documents(
                documents, transformations=[text_splitter]
            )

            # Persist the index
            index.storage_context.persist(persist_dir=index_persist_path)

        total_num_chunks = len(index.docstore.docs)
        print(f"Loaded index with {total_num_chunks} documents")

        return index

    def _create_query_engine(self, chunk_size, chunk_overlap, top_k):
        """Create a query engine with the specified parameters"""
        # Create or load index
        index = self._create_or_load_index(chunk_size, chunk_overlap)

        # Configure retrievers with the specified top_k
        vector_retriever = index.as_retriever(similarity_top_k=top_k, verbose=True)
        bm25_retriever = BM25Retriever.from_defaults(
            docstore=index.docstore, similarity_top_k=top_k
        )

        # Create fusion retriever
        retriever = QueryFusionRetriever(
            [vector_retriever, bm25_retriever],
            similarity_top_k=top_k,
            num_queries=1,  # set to 1 to disable query generation
            mode="reciprocal_rerank",
            use_async=True,
            verbose=True,
        )

        # Create response synthesizer
        response_synthesizer = get_response_synthesizer()

        # Create and return query engine
        return RetrieverQueryEngine(
            retriever=retriever,
            response_synthesizer=response_synthesizer,
        )

    def _add_eval_row(
        self,
        response,
        dataset_example,
        relevancy_result,
        correctness_result,
        response_time,
    ):
        """Add evaluation results to the DataFrame"""
        if not response.source_nodes:
            print("No response!")
            return

        eval_row = pd.DataFrame(
            [
                {
                    "Query": dataset_example.query,
                    "Response": str(response),
                    "Reference Answer": dataset_example.reference_answer,
                    "Source": response.source_nodes[0].node.text[:1000] + "...",
                    "Relevancy Eval Result": f"{'Pass' if relevancy_result.passing else 'Fail'} \nscore: {relevancy_result.score}",
                    "Relevancy Reasoning": relevancy_result.feedback,
                    "Correctness Eval Result": f"{'Pass' if correctness_result.passing else 'Fail'} \n\nscore: {correctness_result.score}",
                    "Correctness Reasoning": correctness_result.feedback,
                    "Response Time": f"{response_time:.4f}",
                }
            ]
        )

        self.eval_results_df = pd.concat(
            [self.eval_results_df, eval_row], ignore_index=True
        )

    async def _evaluate_engine(
        self, query_engine, dataset_examples: LabelledRagDataExample
    ):
        """Evaluate the query engine on the given examples"""
        # # Limit to first 3 examples for testing
        # dataset_examples = dataset_examples[:3]

        # Initialize tracking variables
        relevancy_total_correct = 0
        correctness_total_correct = 0
        correctness_total_score = 0
        total_response_time = 0

        # Process each example
        for dataset_example in dataset_examples:
            start_time = time.time()
            response = query_engine.query(dataset_example.query)
            response_time = time.time() - start_time

            # Evaluate relevancy and correctness
            relevancy_result = self.relevancy_evaluator.evaluate_response(
                query=dataset_example.query, response=response
            )
            correctness_result = self.correctness_evaluator.evaluate_response(
                query=dataset_example.query,
                response=response,
                reference=dataset_example.reference_answer,
            )

            # Add results to DataFrame
            self._add_eval_row(
                response,
                dataset_example,
                relevancy_result,
                correctness_result,
                response_time,
            )

            # Update totals
            total_response_time += response_time
            if relevancy_result.passing:
                relevancy_total_correct += 1
            if correctness_result.passing:
                correctness_total_correct += 1
                correctness_total_score += correctness_result.score

        return (
            relevancy_total_correct,
            correctness_total_correct,
            correctness_total_score,
            len(dataset_examples),
            total_response_time,
        )

    def evaluate_with_params(self, chunk_size, chunk_overlap, top_k):
        """Run evaluation with the specified parameters"""
        print(
            f"Parameters: chunk_size={chunk_size}, chunk_overlap={chunk_overlap}, top_k={top_k}"
        )

        # Reset results DataFrame
        self.eval_results_df = pd.DataFrame()

        # Create query engine with the specified parameters
        query_engine = self._create_query_engine(chunk_size, chunk_overlap, top_k)

        # Run evaluation
        (
            relevancy_total_correct,
            correctness_total_correct,
            correctness_total_score,
            total_questions,
            total_response_time,
        ) = asyncio.run(self._evaluate_engine(query_engine, self.rag_dataset.examples))

        # Display results
        styled_df = self.eval_results_df.style.set_properties(
            **{"white-space": "pre-wrap"},
        )

        display(styled_df)

        # Calculate scores
        relevancy_score = relevancy_total_correct / total_questions
        correctness_score = correctness_total_score / total_questions
        avg_response_time = total_response_time / total_questions

        # Display summary
        print(
            f"Total Relevancy correct: {relevancy_total_correct} out of {total_questions}, "
            f"score: {relevancy_score}"
        )
        print(
            f"Total Correctness correct: {correctness_total_correct} out of {total_questions}, "
            f"score: {correctness_score}"
        )
        print(f"Average response time: {avg_response_time:.4f} seconds")
        print("===============================================")

        # Add result to summary DataFrame
        self._add_summary_row(
            chunk_size,
            chunk_overlap,
            top_k,
            relevancy_score,
            correctness_score,
            avg_response_time,
        )

        return {
            "relevancy_score": relevancy_score,
            "correctness_score": correctness_score,
            "avg_response_time": avg_response_time,
        }

    def _add_summary_row(
        self,
        chunk_size,
        chunk_overlap,
        top_k,
        relevancy_score,
        correctness_score,
        avg_response_time,
    ):
        """Add a summary row to the summary results DataFrame"""
        summary_row = pd.DataFrame(
            [
                {
                    "Chunk Size": chunk_size,
                    "Chunk Overlap": chunk_overlap,
                    "Top K": top_k,
                    "Relevancy Score": f"{relevancy_score:.4f}",
                    "Correctness Score": f"{correctness_score:.4f}",
                    "Avg Response Time": f"{avg_response_time:.4f}",
                }
            ]
        )

        self.summary_results_df = pd.concat(
            [self.summary_results_df, summary_row], ignore_index=True
        )
        self.summary_results_df.to_csv(
            "summary_results_second_half_last_2.csv", index=False
        )

    def run_evaluations(self, chunk_sizes, overlaps, top_ks):
        """Run evaluations for multiple parameter combinations"""
        # Reset summary results DataFrame
        # self.summary_results_df = pd.DataFrame()

        # Generate all parameter combinations
        param_combinations = list(itertools.product(chunk_sizes, overlaps, top_ks))
        total_combinations = len(param_combinations)

        print(f"Running evaluations for {total_combinations} parameter combinations...")

        results = {}
        for i, (chunk_size, overlap, top_k) in enumerate(param_combinations):
            print(f"\nEvaluation {i+1}/{total_combinations}")
            # Convert overlap from percentage to absolute value
            chunk_overlap = int(chunk_size * overlap)
            results[(chunk_size, overlap, top_k)] = self.evaluate_with_params(
                chunk_size, chunk_overlap, top_k
            )

        # Display summary table
        self._display_summary_table()

        return results

    def _display_summary_table(self):
        """Display a summary table of all parameter combinations"""
        print("\n--- Summary of All Parameter Combinations ---")

        # Sort the summary results by scores
        sorted_df = self.summary_results_df.sort_values(
            by=["Relevancy Score", "Correctness Score"], ascending=False
        )

        display(sorted_df)

        # Find the best parameter combination
        best_row = sorted_df.iloc[0]
        print(f"\nBest Parameter Combination:")
        print(f"Chunk Size: {best_row['Chunk Size']}")
        print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
        print(f"Top K: {best_row['Top K']}")
        print(f"Relevancy Score: {best_row['Relevancy Score']}")
        print(f"Correctness Score: {best_row['Correctness Score']}")
        print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")

In [ ]:
[128, 192, 256, 384, 512, 768, 1024, 1536, 2048]

In [4]:
evaluator = RAGEvaluator()

# Define parameter ranges to test
# (512,51,8)~
# (512,25,3)~(512,102,12)
top_ks = [5,7,10]


chunk_sizes = [768]
overlaps = [0.15, 0.2]
results = evaluator.run_evaluations(chunk_sizes, overlaps, top_ks)

chunk_sizes = [1024, 1536, 2048]
overlaps = [0.05, 0.1, 0.15, 0.2]
results = evaluator.run_evaluations(chunk_sizes, overlaps, top_ks)

# Define parameter ranges to test
chunk_sizes = [128, 192, 256, 384]
overlaps = [0.05]
results = evaluator.run_evaluations(chunk_sizes, overlaps, top_ks)

Running evaluations for 6 parameter combinations...

Evaluation 1/6
Parameters: chunk_size=768, chunk_overlap=115, top_k=5
Creating new index for chunk_size=768, chunk_overlap=115
Started parsing the file under job_id 4fe689a7-c111-4cca-a73d-ae0d7b62d0c8
Started parsing the file under job_id 02f06c3f-5d47-4a94-857e-5aa4a96e0ee1
Started parsing the file under job_id f19f06a2-8499-4124-8aa8-dc0e46b0910e
Started parsing the file under job_id b2fb7bb8-b0d2-4505-8464-eeae85ece101
Started parsing the file under job_id 81aa5063-5ec3-4d3e-b7ed-1ee10f66e5de
Started parsing the file under job_id 78cf7a23-6817-48bc-840f-c44a13325fdc
Loaded index with 25 documents


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning,Response Time
0,什麼是學餐資訊？,學餐資訊是一份關於學校食堂的資訊，內容包括了各種不同的餐廳和菜單，強調了SDGs的概念和可持續發展的理念。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,健康學餐（強調SDGs目標3：良好健康與福祉） - (1) 運動員餐廳 - 提供高蛋白、低脂肪的餐點，適合運動後補充能量。 - 招牌菜：健身沙拉、雞胸肉餐盒。 - (2) 心靈健康餐廳 - 提供低糖、低鹽的餐點，支持心理健康。 - 招牌菜：穀物水果碗、香菇豆腐湯。 - (3) 健康果汁吧 - 提供新鮮水果和蔬菜汁，增強免疫力。 - 招牌菜：綠色活力汁、莓果奶昔。...,Pass score: 1.0,YES.,Pass score: 4.0,"The generated answer is relevant to the user query, as it discusses school cafeteria information. However, it contains some inaccuracies and extraneous information (e.g., SDGs concept) that are not present in the reference answer. The answer also seems to be more focused on the concept of sustainability rather than providing concrete information about school meals, which is what the user query is asking for. Despite these issues, the generated answer does touch upon some relevant points, such as different restaurants and menus, so it doesn't deserve a score of 1 or 2.",5.7729
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,校內的學生餐廳採用了 Zero Hunger（零飢餓）、Responsible Consumption and Production（負責任的消費和生產）和Good Health and Well-being（良好健康與福祉）的SDGs目標概念來建立。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,YES.,Pass score: 4.0,"The generated answer is fully relevant to the user query, and it accurately lists the SDGs targets that are adopted by the student cafeteria. The only difference between the generated answer and the reference answer is the formatting of the list items, but this does not affect the correctness or relevance of the answer. Therefore, a score of 4.0 is appropriate.",2.0418
2,什麼是 Green Garden 學餐？,Green Garden 學餐強調的是可持續城市和社區的概念，提供新鮮健康的餐點，並利用校園社區農園的蔬菜。,社區農園餐廳。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,YES.,Pass score: 4.0,"The generated answer is highly relevant to the user query, and it provides a detailed explanation of what Green Garden 學餐 is about. The answer also mentions the use of school community garden's vegetables, which is consistent with the concept of sustainable city and community. However, the answer could be more concise and directly state the definition of Green Garden 學餐, which is mentioned in the reference answer.",2.6676
3,社區農園餐廳招牌菜有哪些？,社區農園餐廳的招牌菜包括社區農園沙拉和蔬菜焗飯。,社區農園沙拉、蔬菜焗飯。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STR

Total Relevancy correct: 27 out of 30, score: 0.9
Total Correctness correct: 29 out of 30, score: 3.9
Average response time: 2.7209 seconds

Evaluation 2/6
Parameters: chunk_size=768, chunk_overlap=115, top_k=7
Loading existing index for chunk_size=768, chunk_overlap=115
Loaded index with 25 documents


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning,Response Time
0,什麼是學餐資訊？,學餐資訊是學校為了倡導可持續發展目標而開設的一系列餐廳，提供健康和環保的食物選擇。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,健康學餐（強調SDGs目標3：良好健康與福祉） - (1) 運動員餐廳 - 提供高蛋白、低脂肪的餐點，適合運動後補充能量。 - 招牌菜：健身沙拉、雞胸肉餐盒。 - (2) 心靈健康餐廳 - 提供低糖、低鹽的餐點，支持心理健康。 - 招牌菜：穀物水果碗、香菇豆腐湯。 - (3) 健康果汁吧 - 提供新鮮水果和蔬菜汁，增強免疫力。 - 招牌菜：綠色活力汁、莓果奶昔。...,Pass score: 1.0,YES.,Pass score: 4.0,"The generated answer is relevant to the user query, as it discusses the concept of ""學餐資訊"". However, it contains some inaccuracies, such as mentioning that the purpose of school cafeteria information is to promote sustainable development goals, which is not mentioned in the reference answer. Despite this, the generated answer still conveys a similar idea and provides relevant information about healthy and environmentally-friendly food options.",2.9810
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,**Rewrite** 校內的學生餐廳採用了未來大學-SDGs永續成果報告.pdf中提到的理念，在惜食餐點發放和有機蔬菜種植等活動中，促進了學校內外的資源共享和可持續發展。這些概念與目標1：零飢餓、目標12：負責任的消費和生產，以及目標3：良好健康與福祉有關，強調了學校餐廳在推動可持續發展方面的重要性。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,- 設立可持續發展教育課程： - 開設選修課程，講授可持續發展和負責任消費的概念。 - 將可持續發展納入必修課程，使每個學生都能接觸到相關知識。 # 目標 13：氣候行動 Climate Action - 設立氣候變遷研究中心 - 成立一個跨學科的研究中心，專注於氣候變遷的研究和創新解決方案。 - 鼓勵學生參與研究項目，並提供實習機會。 - 定期舉辦研討會和學術會議，促進校內外的知識交流。 - 推動校園和社區的環保活動 - 舉辦校園清潔日、植樹活動等，提升學生的環保意識。 - 與當地社區合作，組織環保宣傳活動，如步行和騎行倡議。 # 目標 14：水下生物 Life Below Water - 海洋生態系統的研究和保護工作： - 建立與國際或本地海洋保護組織的合作夥伴關係，共同開展海洋生態研究。 - 組織學生參與海洋保護活動，如海岸清潔和海洋生物監測。 - 在學校內部推廣海洋保護的重要性，並設立相關獎學金支持相關研究。...,Pass score: 1.0,"Based on the provided context and response, I would still answer **YES**. The original question asked if the student cafeteria adopted concepts related to SDGs (Sustainable Development Goals) targets. The response mentioned that the cafeteria promoted resource sharing and sustainable development through activities like reducing food waste and growing organic vegetables, which aligns with SDG 1 (Zero Hunger), SDG 12 (Responsible Consumption and Production), and SDG 3 (Good Health and Well-being). The new context provides additional information about the campus agriculture plan, daily food reduction initiatives, health services, and education programs. While these details are interesting, they do not contradict the original response or question. In fact, some of these initiatives (e.g., reducing food waste) are mentioned in the original response as examples of how the cafeteria promotes sustainable development. Therefore, I would still answer **YES**, as the context information does not change the fact that the student cafeteria adopted concepts related to SDGs targets.",Pass score: 4.0,"The generated answer is highly relevant to the user query, and it accurately describes the concepts of SDGs adopted by the student cafeteria. The answer also provides specific examples and connections to the corresponding SDG targets (1, 12, and 3), demonstrating a thorough understanding of the topic. However, I deduct a small point because the answer is not as concise as the reference answer, which directly lists the relevant SDG targets.",8.0370
2,什麼是 Green Garden 學餐？,**Rewrite** Green Garden 學餐是一個利用校園社區農園的有機蔬菜，提供新鮮健康的餐點。這個計劃不僅讓學生和教職員工能夠享受到營養豐富的食物，也鼓勵大家參與校園農業計劃，培養環保意識和健康生活方式。,社區農園餐廳。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,"YES. The response

Total Relevancy correct: 27 out of 30, score: 0.9
Total Correctness correct: 27 out of 30, score: 3.6
Average response time: 5.3221 seconds

Evaluation 3/6
Parameters: chunk_size=768, chunk_overlap=115, top_k=10
Loading existing index for chunk_size=768, chunk_overlap=115
Loaded index with 25 documents


...

Total Relevancy correct: 28 out of 30, score: 0.9333333333333333
Total Correctness correct: 30 out of 30, score: 4.0
Average response time: 1.5654 seconds

Evaluation 12/12
Parameters: chunk_size=384, chunk_overlap=19, top_k=10
Loading existing index for chunk_size=384, chunk_overlap=19
Loaded index with 51 documents


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning,Response Time
0,什麼是學餐資訊？,學餐資訊是一個倡導SDGs概念的平台，透過提供健康、環保的餐點和活動，促進學生的身心健康和福祉。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,# 學餐資訊 為倡導SDGs的概念，校內的學生餐廳也是採用其中幾個目標的概念來建立： # 22 ZERO HUNGER # 1....,Pass score: 1.0,YES.,Pass score: 4.0,"The generated answer is relevant to the user query, but it contains some incorrect information. The reference answer specifically mentions ""餐點資訊"" (meal information), while the generated answer introduces a new concept of promoting SDGs and activities, which is not mentioned in the user query or reference answer. However, the generated answer does mention ""健康、環保的餐點"" (healthy and eco-friendly meals), which is related to the user query. Therefore, I give it a score of 4 out of 5, indicating that it's mostly correct but with some minor inaccuracies.",1.5856
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,校內的學生餐廳採用了零飢餓（Zero Hunger）、可持續城市和社區（Sustainable Cities and Communities）、負責任的消費和生產（Responsible Consumption and Production）等SDGs目標的概念來建立。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,- 鼓勵同學到食物銀行、關注移工或無家者等非營利組織當志工 # 目標 11：可持續城市和社區 Sustainable Cities and Communities - 校園內實施了多項可持續發展措施，包括綠化校園、建設自行車道和公共交通系統 - 時常舉辦校內二手市集，賦予舊物品新生命 # 目標 12：負責任的消費和生產 Responsible Consumption and Production - 校園內推行廢物分類和回收計劃 - 在校園內設置多個廢物分類站，確保學生和教職員工可以方便地進行廢物分類。 - 設立回收獎勵計劃，鼓勵學生積極參與廢物回收。...,Pass score: 1.0,"YES. The response mentions that the student cafeteria adopts the concepts of Zero Hunger, Sustainable Cities and Communities, and Responsible Consumption and Production (SDGs targets) to establish itself. This is in line with the context information provided, which highlights various initiatives promoting sustainable development and responsible consumption within the school.",Pass score: 4.0,"The generated answer is highly relevant to the user query, as it mentions specific SDGs targets that are related to the establishment of the student cafeteria. The answer also contains accurate information about the adopted SDGs targets, including Zero Hunger and Responsible Consumption and Production. However, the answer does not exactly match the reference answer in terms of formatting and specificity (it includes Sustainable Cities and Communities which is not mentioned in the reference answer), but it still conveys the same overall message. Therefore, a score of 4.0 is appropriate.",1.8087
2,什麼是 Green Garden 學餐？,社區農園餐廳。,社區農園餐廳。,# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5....,Pass score: 1.0,YES,Pass score: 4.0,"The generated answer is identical to the reference answer, indicating that it is fully correct and relevant to the user query. The conciseness of the answer is also similar, making it a strong candidate for a high score.",1.2435
3,社區農園餐廳招牌菜有哪些？,社區農園餐廳的招牌菜包括社區農園沙拉和蔬菜焗飯。,社區農園沙拉、蔬菜焗飯。,# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5....,Pass score: 1.0,"YES. The response for the query ""社區農園餐廳招牌菜有哪些？"" is in line with the context information provided, as it mentions specific menu items that match the description of the Green Garden restaurant's signature dishes.",Pass score: 4.0,"The generated answer is fully correct and relevant to the user query, with no mistakes or omissions. It even provides a bit more detail than the reference answer by explicitly stating that the dishes are ""招牌菜"" (signature dishes) of the community garden restaurant. The only reason I wouldn't give it a perfect score of 5.0 is that it's not significantly more concise or informative than the reference answer, but overall it's a very good response.",1.4703
4,未來大學（Futu

Total Relevancy correct: 30 out of 30, score: 1.0
Total Correctness correct: 29 out of 30, score: 3.9
Average response time: 1.5728 seconds

--- Summary of All Parameter Combinations ---


,Chunk Size,Chunk Overlap,Top K,Relevancy Score,Correctness Score,Avg Response Time
53,384,19,10,1.0000,3.9000,1.5728
51,384,19,5,1.0000,3.8667,1.8271
18,1536,76,5,1.0000,3.3667,4.3494
39,2048,409,5,1.0000,3.2000,3.7722
26,1536,230,10,1.0000,2.8000,10.0441
41,2048,409,10,1.0000,2.8000,11.9728
12,1024,153,5,0.9667,3.9333,1.9838
6,1024,51,5,0.9667,3.8667,1.9859
27,1536,307,5,0.9667,3.8667,6.6885
46,192,9,7,0.9667,3.8000,1.8975



Best Parameter Combination:
Chunk Size: 384
Chunk Overlap: 19
Top K: 10
Relevancy Score: 1.0000
Correctness Score: 3.9000
Avg Response Time: 1.5728 seconds
